# L3: 数学基础回顾


# L3: 数学基础回顾

**课时**: 2 课时（90 分钟/课时）

## 1. 学习目标

完成本课程后，学员能够：

1. 理解向量、矩阵的几何意义，并能用 NumPy 进行基本线性代数运算（加减乘、转置、逆、特征值）
2. 解释概率论中的条件概率、贝叶斯定理，并应用于简单场景的推理计算
3. 说明信息熵、交叉熵的数学含义，理解它们在 ML 损失函数中的应用
4. 能读懂并手动推导梯度下降的单步更新公式
5. 对积分、导数、极值有直观理解，能识别机器学习中的数学符号

## 2. 核心知识点

### 2.1 线性代数

**向量与矩阵基础**

- **向量**: n 维有序数列表，几何上表示空间中的一个点或方向
- **矩阵**: m×n 数表，表示线性变换（旋转、缩放、投影）
- **核心运算**:
  - 加减：对应元素操作
  - 数乘：每个元素乘以标量
  - 矩阵乘法：(m×n)·(n×p) = (m×p)，复杂度 O(n²)
  - 转置：行列互换，记作 Aᵀ
  - 逆矩阵：A⁻¹，满足 AA⁻¹ = I（仅方阵且行列式非零时存在）

**神经网络中的线性代数**

- 输入向量 x ∈ ℝⁿ 通过权重矩阵 W ∈ ℝⁿˣᵐ 变换为隐向量 h = Wx + b
- 批量处理时使用矩阵乘法一次完成多个样本的计算（GPU 并行基础）


In [ ]:
import numpy as np

W = np.array([[3, 1], [2, 4]])  # 2x2 权重矩阵
x = np.array([1, 2])            # 输入向量
b = np.array([0.5, -0.5])      # 偏置

h = W @ x + b                  # 前向传播（矩阵乘向量）
print(f"隐层输出: {h}")        # [3*1+1*2+0.5, 2*1+4*2-0.5] = [5.5, 7.5]

# 批量处理（3个样本）
X_batch = np.array([[1, 2], [3, 4], [5, 6]])  # 3x2
H_batch = X_batch @ W.T + b                   # 3x2
print(f"批量输出:\n{H_batch}")


### 2.2 概率统计

**基本概念**

- **随机变量**: 取值随机的变量，分为离散（如骰子）和连续（如身高）
- **概率分布**: 描述随机变量取各值的概率（P(X=x)）
- **期望值 E[X]**: 随机变量的长期平均值（加权求和）
- **方差 Var(X)**: 数据离散程度的度量 E[(X-E[X])²]

**条件概率与贝叶斯定理**

- P(A|B) = P(A∩B) / P(B)（B 发生时 A 的概率）
- **贝叶斯定理**: P(A|B) = P(B|A)·P(A) / P(B)
  - P(A) 先验概率
  - P(A|B) 后验概率
  - P(B|A) 似然度


In [ ]:
# 贝叶斯定理示例：医疗诊断
# 某疾病患病率 P(疾病)=0.01（1%）
# 检验准确率 P(阳性|疾病)=0.99，P(阴性|无病)=0.99

P_disease = 0.01
P_positive_given_disease = 0.99
P_negative_given_healthy = 0.99

P_positive_given_healthy = 1 - P_negative_given_healthy  # 0.01

# P(阳性) = P(阳性|疾病)P(疾病) + P(阳性|健康)P(健康)
P_positive = P_positive_given_disease * P_disease + P_positive_given_healthy * (1 - P_disease)

# P(疾病|阳性) = P(阳性|疾病)P(疾病) / P(阳性)
P_disease_given_positive = P_positive_given_disease * P_disease / P_positive

print(f"P(阳性) = {P_positive:.4f}")
print(f"P(疾病|阳性) = {P_disease_given_positive:.4f}")  # 约 0.5（检验呈阳性时患病概率仅50%！）


**统计在 ML 中的应用**

- 线性回归：最小二乘法 → 最小化均方误差
- 分类：逻辑回归使用 Sigmoid 将线性输出转为概率
- 正则化：贝叶斯视角下的先验假设（岭回归 = 高斯先验）

### 2.3 信息论

**熵（Entropy）**

- H(X) = -∑ p(x) log₂ p(x)（单位：比特）
- 描述随机变量的不确定性；均匀分布熵最大

**交叉熵（Cross-Entropy）**

- CE(p, q) = -∑ p(x) log q(x)
- 机器学习中作为分类损失函数（Softmax + Cross-Entropy）
- 交叉熵 ≥ 熵，当 q=p 时等于熵


In [ ]:
import numpy as np

def entropy(p):
    """计算离散分布的香农熵（以2为底，单位比特）"""
    p = np.array(p)
    # 过滤掉概率为0的情况（0*log(0)定义为0）
    return -np.sum(p * np.log2(p + 1e-10))

def cross_entropy(p, q):
    """计算分布p对分布q的交叉熵"""
    p, q = np.array(p), np.array(q)
    return -np.sum(p * np.log2(q + 1e-10))

# 示例：抛硬币（均匀分布）
p_coin = [0.5, 0.5]
print(f"硬币熵: {entropy(p_coin):.4f} bits")  # 1.0 bits

# 偏置硬币
p_biased = [0.9, 0.1]
print(f"偏置硬币熵: {entropy(p_biased):.4f} bits")  # < 1.0 bits

# 交叉熵（用均匀分布q描述偏置硬币）
q_uniform = [0.5, 0.5]
print(f"交叉熵 H(p,q): {cross_entropy(p_biased, q_uniform):.4f} bits")
print(f"熵 H(p): {entropy(p_biased):.4f} bits")
print(f"差异 = {cross_entropy(p_biased, q_uniform) - entropy(p_biased):.4f} bits（KL散度）")


### 2.4 微积分基础

**导数与梯度**

- 导数：函数在某点的瞬时变化率（切线斜率）
- 梯度：多元函数在每维上的偏导数向量 ▽f = [∂f/∂x₁, ∂f/∂x₂, ...]
- 梯度指向函数上升最快的方向；沿负梯度方向下降最快

**梯度下降算法（手动推导）**

目标：最小化均方误差 MSE = (1/n)∑(ŷᵢ - yᵢ)²


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# 梯度下降演示：一维函数 f(x) = x^2 + 2x + 1 的最小化
def f(x):
    return x**2 + 2*x + 1

def df(x):
    """f(x)的解析导数：2x + 2"""
    return 2*x + 2

# 梯度下降
x = 10.0          # 初始点
lr = 0.1          # 学习率
trajectory = [x]

for step in range(50):
    grad = df(x)
    x = x - lr * grad
    trajectory.append(x)

print(f"最优点 x ≈ {x:.6f}")
print(f"最小值 f(x) ≈ {f(x):.6f}")
print(f"解析最优点 x* = -1（由2x+2=0得）")

# 可视化
xs = np.linspace(-12, 12, 300)
plt.figure(figsize=(10, 5))
plt.plot(xs, f(xs), label="f(x) = x² + 2x + 1", color="#2c3e50")
plt.plot(trajectory, [f(x) for x in trajectory], "o-", color="#e74c3c",
         label="梯度下降路径", markersize=4)
plt.axvline(x=-1, color="green", linestyle="--", label="解析最优点 x*=-1")
plt.xlabel("x")
plt.ylabel("f(x)")
plt.title("梯度下降算法示意图")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()


## 3. 代码示例


In [ ]:
"""
L3-math-review.py
综合演示：线性代数 + 概率 + 信息论
"""

import numpy as np
import matplotlib.pyplot as plt

print("=" * 50)
print("L3 数学基础综合演示")
print("=" * 50)

# ---- Part 1: 矩阵分解与特征值 ----
print("\n[1] 特征值分解：PCA 的数学核心")
A = np.array([[2, 1], [1, 2]])
eigenvalues, eigenvectors = np.linalg.eig(A)
print(f"矩阵 A = {A.tolist()}")
print(f"特征值 λ = {eigenvalues.round(4)}")
print(f"特征向量 v = \n{eigenvectors.round(4)}")
# 验证 Av = λv
v = eigenvectors[:, 0]
print(f"验证 Av = λv: {np.allclose(A @ v, eigenvalues[0] * v)}")

# ---- Part 2: 贝叶斯推断 ----
print("\n[2] 贝叶斯推断：垃圾邮件分类基础")
# P(垃圾|包含"免费") = P("免费"|垃圾)P(垃圾) / P("免费")
# 假设：P(垃圾)=0.3, P("免费"|垃圾)=0.6, P("免费"|正常)=0.05
# P("免费") = 0.6*0.3 + 0.05*0.7 = 0.18+0.035 = 0.215
p_spam = 0.3
p_free_given_spam = 0.6
p_free_given_healthy = 0.05
p_free = p_free_given_spam * p_spam + p_free_given_healthy * (1 - p_spam)
p_spam_given_free = p_free_given_spam * p_spam / p_free
print(f"P(垃圾|"免费") = {p_spam_given_free:.4f} (约 {p_spam_given_free*100:.1f}%)")

# ---- Part 3: 交叉熵损失函数可视化 ----
print("\n[3] 交叉熵 vs 均方误差：分类任务中的表现")
p_true = np.array([0, 1])  # 真实分布：类别1概率100%
ce_values, mse_values = [], []

for q1 in np.linspace(0.01, 0.99, 50):
    q = np.array([1 - q1, q1])
    ce = -np.sum(p_true * np.log(q + 1e-10))
    mse = np.mean((p_true - q) ** 2)
    ce_values.append(ce)
    mse_values.append(mse)

plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
plt.plot(ce_values, label="Cross-Entropy", color="#3498db")
plt.plot(mse_values, label="MSE", color="#e74c3c", linestyle="--")
plt.xlabel("预测分布 q₁ 的变化")
plt.ylabel("损失值")
plt.title("分类损失函数对比")
plt.legend()
plt.grid(True, alpha=0.3)

plt.subplot(1, 2, 2)
p_range = np.linspace(0.01, 0.99, 100)
h = -p_range * np.log2(p_range) - (1 - p_range) * np.log2(1 - p_range + 1e-10)
plt.plot(p_range, h, color="#2ecc71")
plt.xlabel("p(X=1)")
plt.ylabel("H(p)")
plt.title("二元分布的香农熵")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("l3_math_demo.png", dpi=150)
plt.show()


## 4. 练习题

1. **线性代数**: 给定矩阵 A = [[1,2],[3,4]] 和向量 x = [1,1]，计算 A·x、Aᵀ、A 的行列式和逆矩阵（使用 NumPy 验证手算结果）。
2. **概率计算**: 一种疾病检测方法的准确率为 99%（患病者检测阳性概率 99%，未患病者检测阴性概率 99%）。若该疾病在人群中的患病率为 0.5%，计算检测结果为阳性时真正患病的概率。
3. **信息论**: 计算抛掷一枚均匀六面骰子的香农熵（以比特为单位），然后计算一枚偏置骰子（6 出现概率为 50%，其他面各 10%）的熵，说明为何偏置分布熵更小。
4. **梯度下降**: 对于函数 f(x) = x⁴ - 4x²，手动推导其梯度公式，并使用 Python 实现梯度下降算法，找到一个局部最小值点（初始值 x₀=0.5，学习率 0.01，迭代 100 次）。
5. **综合应用**: 假设一个神经网络最后一层的原始输出为 logit = [2.0, 1.0, 0.5]，请手动计算：① 通过 Softmax 转为概率分布；② 假设真实标签为类别 0（one-hot 为 [1,0,0]），计算交叉熵损失值。

## 5. 延伸阅读

- **书籍**: Gilbert Strang, *Linear Algebra and Its Applications*（第5版）— 麻省理工线性代数教材，配套视频可在 MIT OpenCourseWare 免费观看
- **书籍**: 西格蒙德, 《概率论与数理统计》— 国内经典教材
- **在线课程**: 3Blue1Brown, *Essence of Linear Algebra*（B站中文翻译）— 几何直觉理解线性代数
- **在线课程**: Khan Academy, *Statistics & Probability* — 互动式概率统计学习
- **工具**: https://www.geogebra.org/ — 可视化数学概念（梯度、矩阵变换等）

---
